#  03. CRAWL DỮ LIỆU KHÍ HẬU & THỜI TIẾT (NASA POWER API)
> **Mục đích:** Crawl dữ liệu thời tiết hàng ngày từ NASA POWER Agroclimatology API (Nhiệt độ max/min/mean, Lượng mưa, Độ ẩm, Gió) và tính Số ngày stress nhiệt (>35°C).


### 3.1 Import Thư viện & Phân loại Mùa vụ


In [ ]:
import os
import sys

if sys.stdout.encoding != 'utf-8':
    sys.stdout.reconfigure(encoding='utf-8')

import requests
import pandas as pd
import numpy as np
from datetime import datetime
from time import sleep

COORDS_FILE = os.path.join(os.path.dirname(__file__), "Bangladesh_districts_coords_data.csv")
MAIN_DATA_FILE = os.path.join(os.path.dirname(__file__), "Bangladesh_main_data.csv")
OUTPUT_FILE = os.path.join(os.path.dirname(__file__), "Process_Bangladesh_weather_data_Merge.csv")
YEAR = 2022

def get_season(month):
    if month in [11, 12, 1, 2]: return "Rabi"
    if month in [3, 4, 5, 6]: return "Kharif 1"
    if month in [7, 8, 9, 10]: return "Kharif 2"
    return None


### 3.2 Hàm Gọi NASA POWER API & Chuyển đổi DataFrame


In [ ]:
def fetch_nasa(lat, lon):
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        "?parameters=PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,WS2M,WS2M_MAX"
        "&community=AG"
        f"&latitude={lat}&longitude={lon}"
        f"&start={YEAR}0101&end={YEAR}1231&format=JSON"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["properties"]["parameter"]

def nasa_to_df(data):
    rows = [{
        "Date": datetime.strptime(d, "%Y%m%d"),
        "Rainfall": data["PRECTOTCORR"][d],
        "Temp_Mean": data["T2M"][d],
        "Temp_Max": data["T2M_MAX"][d],
        "Temp_Min": data["T2M_MIN"][d],
        "Wind_Mean": data["WS2M"][d],
        "Wind_Max": data["WS2M_MAX"][d],
    } for d in data["T2M"]]
    
    df = pd.DataFrame(rows)
    df["Month"] = df["Date"].dt.month
    df["Season"] = df["Month"].apply(get_season)
    return df


### 3.3 Hàm Gom nhóm Theo Mùa vụ & Tính Ngày Stress Nhiệt (>35°C)


In [ ]:
def aggregate_season(df, district):
    out = df.groupby("Season").agg(
        Rainfall=("Rainfall", "sum"),
        Temp_Mean=("Temp_Mean", "mean"),
        Temp_Max=("Temp_Max", "max"),
        Temp_Min=("Temp_Min", "min"),
        Heat_Stress_Days=("Temp_Max", lambda x: (x > 35).sum()),
        Wind_Mean=("Wind_Mean", "mean"),
        Wind_Max=("Wind_Max", "max")
    ).reset_index()
    out["District"] = district
    out["Year"] = YEAR
    return out


### 3.4 Hàm Chạy Vòng lặp Crawl Thời tiết


In [ ]:
def crawl_data():
    try:
        coords = pd.read_csv(COORDS_FILE)
    except FileNotFoundError:
        return None

    all_data = []
    total = len(coords)
    for i, row in coords.iterrows():
        name = row.get("District", row.get("district", "Unknown"))
        print(f"[{i+1}/{total}] Đang tải dữ liệu thời tiết cho khu vực: {name}...")
        try:
            raw = fetch_nasa(row["lat"], row["lon"])
            daily = nasa_to_df(raw)
            seasonal = aggregate_season(daily, name)
            all_data.append(seasonal)
        except Exception:
            print(f"  -> Lỗi không lấy được dữ liệu cho {name}")
            pass
        sleep(1)

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return None


### 3.5 Làm sạch & Feature Engineering (Rain/Temp Ratio)


In [ ]:
def clean_data(df):
    df['District'] = df['District'].astype(str).str.strip().str.title()
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isna().sum() > 0:
            df[col] = df.groupby('District')[col].transform(lambda x: x.fillna(x.median()))
    return df

def feature_engineering(df):
    df['Rain_Temp_Ratio'] = (df['Rainfall'] / (df['Temp_Mean'] + 0.001)).round(2)
    return df


### 3.6 Thực thi Pipeline & Lưu File CSV


In [ ]:
def merge_and_save(df):
    try:
        main_df = pd.read_csv(MAIN_DATA_FILE)
        valid_districts = main_df['District'].dropna().unique()
        missing = set(df['District']) - set(valid_districts)
    except FileNotFoundError:
        pass
        
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

if __name__ == "__main__":
    print(f"Bắt đầu quy trình xử lý Weather...")
    df_raw = crawl_data()
    if df_raw is not None and not df_raw.empty:
        df_clean = clean_data(df_raw)
        df_final = feature_engineering(df_clean)
        merge_and_save(df_final)
        print(f"Đã lưu kết quả tại {OUTPUT_FILE}")
    else:
        print("Không có dữ liệu.")
